In [1]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler
import polars
from tqdm import tqdm
import lsst.geom as geom
import pandas as pd
import matplotlib.colors as colors

In [2]:
repo = "dp2_prep"
collection = "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2"

PARQUET_COLUMNS = [
    'slot_Centroid_x', 'slot_Centroid_y', 'detector', 'psf_max_value',
]

butler = Butler(repo, collections=collection)

In [3]:
def getStarImage(visit, detector, stampSize = 41):

#stampSize = 41
#visit = 2025050400608
#detector = 42
    
    dataID = {
        "instrument": "LSSTCam", 
        "visit": visit,
        "detector": detector,
    }
    uri = butler.getURI("refit_psf_star", **dataID)
    parquet_path = uri.geturl()
    psfTable = polars.scan_parquet(parquet_path).select(PARQUET_COLUMNS).collect()
    psfTable = psfTable.filter(polars.col("detector") == detector)
    
    calexp = butler.get("preliminary_visit_image", **dataID)
    VMIN = 1e-5
    VMAX = 1e-1
    try: 
        plt.figure(figsize=(8,8))
        plt.subplots_adjust(wspace=0.)
        stars = []
        weights = [] 
        for i in range(9):
            positionStar = geom.Point2D(psfTable['slot_Centroid_x'][i], psfTable['slot_Centroid_y'][i])
            srcImg = calexp.getCutout(positionStar, geom.Extent2I(stampSize, stampSize))
            im = srcImg.getMaskedImage()
            star = im.image.array / np.sum(im.image.array)
            plt.subplot(3,3,i+1)
            plt.imshow(star, cmap=plt.cm.Greys_r, vmin=0, vmax=np.max(star))
            plt.gca().invert_yaxis()
            plt.xticks([],[])
            plt.yticks([],[])
            if i == 1:
                plt.title(f"{visit} | {detector}")
    except:
        print('failed')

In [4]:
badVisitDic = pd.read_pickle('data/badDetectorList.pkl')
zernike_table =  pd.read_pickle("data/visit_to_band_mapv2.pkl")

visits = np.array([visit for visit in badVisitDic if visit in zernike_table])
z4 = np.array([np.nanmedian(zernike_table[visit]['z4']) for visit in visits])
FiltreBad = abs(z4+0.25)>1.5
print(np.sum(FiltreBad))

263


In [ ]:
N_bad = 0 

for visit in visits[FiltreBad]:
    getStarImage(visit, 176, stampSize = 41)
    N_bad += 1
    if N_bad > 25:
        break

In [ ]:
#zernikeDic = {}
#for visit in zernike_table:
#        if visit in visitsDP2 and zernike_table[visit]['band'] in bands:
#            if visit not in zernikeDic:
#                zernikeDic[visit] = {
#                    'zernike': zernike_table[visit][zernikeKey],
#                    'band': zernike_table[visit]['band'],
#                }

    # Get all Zernike values for histogram
#    z_all = [np.median(zernikeDic[visit]['zernike']) for visit in zernikeDic]

In [ ]:
np.nanmedian(zernike_table[2025050400608]['z4'])

In [ ]:
visits = np.array([visit for visit in badVisitDic if visit in zernike_table])
z4 = np.array([np.nanmedian(zernike_table[visit]['z4']) for visit in visits])

In [ ]:
FiltreBad

In [6]:
zernike_table[2025042500576]['z4']

[-0.7890902757644653,
 -1.5207877159118652,
 0.08718975633382797,
 -0.7780255675315857]